# `losses.py` — Custom Loss Functions for Class Imbalance

## Purpose
Defines the loss functions used to train the multi-aspect sentiment model on the **severely imbalanced** cosmetics review dataset.

## Why custom losses?
Standard Cross-Entropy loss treats all classes equally. When the training set has 2,244 positive `price` samples but only 17 negative, the model learns to always predict `positive` for `price` and still achieves ~99% accuracy — but completely fails on the minority classes. The losses in this file are specifically designed to counter this.

## Loss Functions

| Class | Key idea |
|-------|----------|
| `FocalLoss` | Down-weights easy (well-classified) samples so the model focuses on hard, rare ones |
| `ClassBalancedLoss` | Reweights by **effective number of samples** (not raw count) to avoid over-penalising tiny classes |
| `DiceLoss` | Optimises F1-score directly instead of log-likelihood |
| `HybridLoss` | Combines all three above with configurable coefficients |
| `AspectSpecificLossManager` | Assigns different hyperparameters per aspect based on its imbalance severity |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

## 1. Focal Loss
Modifies cross-entropy so that already-correctly-classified examples have **less weight**. The `gamma` parameter controls this — the higher the gamma, the more the model focuses on hard/misclassified examples.

The formula is: `FL(p_t) = -(1 - p_t)^gamma * log(p_t)`  
When `p_t` is high (easy example), `(1 - p_t)^gamma` becomes very small → low weight. When `p_t` is low (hard example) → higher weight.

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss for addressing class imbalance.

    Args:
        alpha: Per-class weighting tensor. If None, all classes get equal weight.
        gamma: Focusing parameter. gamma=0 → standard CE. gamma=2 is typical.
        reduction: 'mean' (default), 'sum', or 'none'
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma     = gamma       # Controls how much to focus on hard examples
        self.alpha     = alpha       # Per-class weight (inverse-frequency weights usually)
        self.reduction = reduction

    def forward(self, inputs, targets):
        """
        Args:
            inputs:  (batch_size, num_classes) — raw logits from the model
            targets: (batch_size,) — integer class labels
        """
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')  # Per-sample CE loss
        pt      = torch.exp(-ce_loss)                                 # pt = model's confidence on the correct class

        focal_loss = (1 - pt) ** self.gamma * ce_loss  # Down-weight easy samples

        if self.alpha is not None:
            # Map per-class alpha weights to per-sample weights using target indices
            if isinstance(self.alpha, (list, np.ndarray)):
                alpha_t = torch.tensor(self.alpha, device=inputs.device, dtype=torch.float)[targets]
            elif isinstance(self.alpha, torch.Tensor):
                alpha_t = self.alpha.to(inputs.device)[targets]
            else:
                alpha_t = self.alpha   # Scalar — same weight for all samples
            focal_loss = alpha_t * focal_loss

        if self.reduction == 'mean': return focal_loss.mean()
        elif self.reduction == 'sum': return focal_loss.sum()
        return focal_loss

## 2. Class-Balanced Loss

Instead of weighting by `1/n_i` (raw inverse frequency), this uses the **Effective Number of Samples** formula:  
`E_n = (1 - beta^n) / (1 - beta)`  

As `n` grows (majority class), `E_n` saturates, so the effective weight for a class with 2,000 samples isn't 100x less than one with 20 — it's much more moderate. This avoids catastrophically over-weighting tiny classes.

In [ ]:
class ClassBalancedLoss(nn.Module):
    """
    Class-Balanced Loss using Effective Number of Samples.
    
    Args:
        samples_per_class: List [neg_count, neu_count, pos_count]
        beta: Controls saturation. 0.9999 for extreme imbalance, 0.999 for moderate.
    """
    def __init__(self, samples_per_class, beta=0.9999, reduction='mean'):
        super(ClassBalancedLoss, self).__init__()

        # Effective number of samples for each class
        effective_num = 1.0 - np.power(beta, samples_per_class)   # E_n = 1 - beta^n
        weights       = (1.0 - beta) / np.array(effective_num)    # Normalise by (1-beta)

        # Re-normalise so average weight = 1.0 (keeps the loss scale the same as CE)
        weights = weights / weights.sum() * len(weights)

        self.weights   = torch.tensor(weights, dtype=torch.float32)
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Move weights to the same device as inputs (GPU if available)
        if self.weights.device != inputs.device:
            self.weights = self.weights.to(inputs.device)
        # Standard CrossEntropy, but weighted by the effective-number weights
        return F.cross_entropy(inputs, targets, weight=self.weights, reduction=self.reduction)

## 3. Dice Loss
Directly optimises the **Dice coefficient** (equivalent to F1-score). This is especially helpful for imbalanced classification because it does NOT depend on the absolute size of each class — it only cares about precision and recall on each class equally.

In [ ]:
class DiceLoss(nn.Module):
    """Dice Loss: directly optimises the F1-score (Dice coefficient)."""
    def __init__(self, smooth=1.0):
        super(DiceLoss, self).__init__()
        self.smooth = smooth  # Laplace smoothing to avoid division by zero

    def forward(self, inputs, targets):
        inputs                = F.softmax(inputs, dim=1)  # Convert logits to probabilities
        targets_one_hot       = F.one_hot(targets, num_classes=inputs.size(1)).float()  # Binary indicator matrix

        intersection = (inputs * targets_one_hot).sum(dim=0)  # True-positive-like term, per-class
        cardinality  = inputs.sum(dim=0) + targets_one_hot.sum(dim=0)  # Predicted + actual totals

        # Dice per class = 2*TP / (Predicted + Actual), then average across classes
        dice_score = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return 1.0 - dice_score.mean()  # Loss = 1 - Dice (minimise → maximise F1)

## 4. Hybrid Loss
Combines FocalLoss + ClassBalancedLoss + DiceLoss with configurable scalar weights.

The final Ablation A7 winner uses `{'focal': 1.0, 'cb': 0.5, 'dice': 0.0}` — Dice was dropped as it didn't help after the CB loss was already rebalancing classes.

In [ ]:
class HybridLoss(nn.Module):
    """
    Weighted sum of Focal + Class-Balanced + Dice losses.
    The `weights` dict configures the contribution of each component.
    """
    def __init__(self, samples_per_class, focal_alpha=None, focal_gamma=2.0,
                 cb_beta=0.9999, weights=None):
        super(HybridLoss, self).__init__()

        if weights is None:
            weights = {'focal': 1.0, 'cb': 0.5, 'dice': 0.3}  # Default from ablation A3

        self.focal_loss = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)
        self.cb_loss    = ClassBalancedLoss(samples_per_class, beta=cb_beta)
        self.dice_loss  = DiceLoss()
        self.weights    = weights

    def forward(self, inputs, targets):
        loss_focal = self.focal_loss(inputs, targets)   # Focus on hard/rare examples
        loss_cb    = self.cb_loss(inputs, targets)      # Reweight by effective sample count
        loss_dice  = self.dice_loss(inputs, targets)    # Directly optimise F1

        total_loss = (
            self.weights.get('focal', 0.0) * loss_focal +
            self.weights.get('cb',    0.0) * loss_cb    +
            self.weights.get('dice',  0.0) * loss_dice
        )

        loss_dict = {
            'focal': loss_focal.item(),  # Individual components for logging
            'cb'   : loss_cb.item(),
            'dice' : loss_dice.item(),
            'total': total_loss.item(),
        }
        return total_loss, loss_dict

## 5. AspectSpecificLossManager
Creates a **separate `HybridLoss` for each aspect**, automatically tuning `gamma` and `beta` based on how imbalanced that aspect's class distribution is.

- `price` has ~130:1 imbalance (positive vs negative) → gets `gamma=3.0`, `beta=0.9999`
- `smell` has more moderate imbalance → gets `gamma=2.0`, `beta=0.999`

This fine-grained tuning was critical for getting the model to predict rare classes at all.

In [ ]:
class AspectSpecificLossManager:
    """Creates one HybridLoss per aspect, auto-tuned by its imbalance severity."""

    def __init__(self, aspect_class_counts, config):
        """
        Args:
            aspect_class_counts: dict {aspect_name: [neg_count, neu_count, pos_count]}
            config: the full training config dict (reads loss_weights etc.)
        """
        self.aspect_losses = {}

        for aspect, counts in aspect_class_counts.items():
            max_count      = max(counts)
            min_count      = min(counts)
            imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')

            # Auto-select gamma: more severe imbalance → higher gamma → stronger focus
            if imbalance_ratio > 50: gamma = 3.0   # e.g. price, packing
            elif imbalance_ratio > 10: gamma = 2.5 # e.g. colour
            else: gamma = 2.0                      # e.g. stayingpower, texture

            # Auto-select beta for Class-Balanced loss
            beta = 0.9999 if imbalance_ratio > 50 else 0.999

            # Compute focal alpha from inverse frequency: rare class gets higher weight
            total       = sum(counts)
            focal_alpha = [total / (len(counts) * c) if c > 0 else 1.0 for c in counts]
            focal_alpha = torch.tensor(focal_alpha, dtype=torch.float32)

            # Build the HybridLoss for this aspect
            self.aspect_losses[aspect] = HybridLoss(
                samples_per_class=counts,
                focal_alpha=focal_alpha,
                focal_gamma=gamma,
                cb_beta=beta,
                weights=config.get('loss_weights', {'focal': 1.0, 'cb': 0.5})
            )

    def compute_loss(self, predictions, targets, aspect_ids, aspect_names):
        """
        Compute per-sample loss using each sample's aspect-specific loss function.
        Returns the mean loss over the batch and a dict with per-aspect breakdowns.
        """
        total_loss  = 0
        loss_details = {}

        for i in range(predictions.size(0)):
            aspect     = aspect_names[aspect_ids[i].item()]  # e.g. 'price'
            loss_fn    = self.aspect_losses[aspect]           # That aspect's HybridLoss
            sample_loss, loss_dict = loss_fn(
                predictions[i].unsqueeze(0),   # Single-sample batch
                targets[i].unsqueeze(0)
            )
            total_loss += sample_loss

            # Accumulate per-aspect loss history for logging
            if aspect not in loss_details:
                loss_details[aspect] = {}
            for key, val in loss_dict.items():
                loss_details[aspect].setdefault(key, []).append(val)

        total_loss = total_loss / predictions.size(0)  # Average over batch

        # Average per-step losses for each aspect key
        for aspect in loss_details:
            for key in loss_details[aspect]:
                loss_details[aspect][key] = np.mean(loss_details[aspect][key])

        return total_loss, loss_details

## Quick Test

In [ ]:
# Simulate a severely imbalanced aspect like 'price': mostly positive (2244), few negative (17)
samples_per_class = [17, 15, 2244]  # [neg, neu, pos]
batch_size, num_classes = 8, 3

predictions = torch.randn(batch_size, num_classes)          # Random logits
targets     = torch.tensor([2, 2, 2, 2, 0, 2, 1, 2])       # Mostly positive

# Test each loss individually
print('FocalLoss:         ', FocalLoss(gamma=3.0)(predictions, targets).item())
print('ClassBalancedLoss: ', ClassBalancedLoss(samples_per_class)(predictions, targets).item())
print('DiceLoss:          ', DiceLoss()(predictions, targets).item())

loss, breakdown = HybridLoss(samples_per_class)(predictions, targets)
print('HybridLoss total:  ', loss.item())
print('Breakdown:', breakdown)